In [1]:
import torch
import torch.nn as nn

In [21]:
class LSTM(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
        super().__init__()
        self.hidden_size=hidden_size
        self.output_size=output_size
        self.input_size=input_size
        self.gate=nn.Linear(self.input_size+self.hidden_size,self.hidden_size*4)
        self.fc=nn.Linear(self.hidden_size,self.output_size)
        self.tanh=nn.Tanh()
    def forward(self,x):
        batch_size,seq_len,input_size=x.shape
        h=torch.zeros(batch_size,self.hidden_size)
        c=torch.zeros(batch_size,self.hidden_size)
        outputs=[]
        for i in range(seq_len):
            x_t=x[:,i,:]
            combined=torch.cat((x_t,h),dim=1)
            gates=self.gate(combined)
            i_gate,f_gate,g_gate,o_gate=gates.chunk(4,dim=1)
            i_gate=torch.sigmoid(i_gate)
            f_gate=torch.sigmoid(f_gate)
            g_gate=self.tanh(g_gate)
            o_gate=torch.sigmoid(o_gate)
            c=f_gate*c+i_gate*g_gate
            h=o_gate*self.tanh(c)
        out=self.fc(h)
        return out
            
        
    

In [24]:
batch_size=10
seq_len=256
input_size=100
hidden_size=512
output_size=10
lstm=LSTM(input_size,hidden_size,output_size)
x=torch.randn(batch_size,seq_len,input_size)
output=lstm(x)
# 正确的断言方式
print(f"输入形状: {x.shape}")      # torch.Size([10, 256, 100])
print(f"输出形状: {output.shape}") # torch.Size([10, 10])

# 检查 Batch Size 是否保留
assert output.shape[0] == x.shape[0]
# 检查输出维度是否为定义的 output_size
assert output.shape[1] == output_size

print("✅ 逻辑正确：模型成功将长序列压缩为了分类向量！")

输入形状: torch.Size([10, 256, 100])
输出形状: torch.Size([10, 10])
✅ 逻辑正确：模型成功将长序列压缩为了分类向量！
